<a href="https://colab.research.google.com/github/adhishmodi/customer-churn-prediction/blob/main/notebooks/08_final_model_selection_and_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Milestone 5 — Phase 4: Final Model Selection and Evaluation

This notebook freezes the final candidate using the **training-only Phase 3 outer-fold results**, fits its calibrator and decision threshold using training-only out-of-fold (OOF) predictions, then evaluates the frozen model on the original 20% holdout.

**Leakage controls**
- Phase 3 outer-fold F1 is the selection metric; precision is the first tie-breaker.
- The held-out test set is not used to select the model, calibrator, or threshold.
- The final threshold is learned from training-only OOF predictions.
- Test metrics are calculated only after the final model and threshold are frozen.
- The same holdout was previously used for the Milestone 4 historical baseline. This is a final comparison, not a fresh untouched test set. Do not tune anything after viewing these results.

## 1. Runtime setup and required files

Run from Google Colab. You need the original IBM Telco CSV and the Phase 3 fold-score CSV downloaded from your Phase 3 run.

In [9]:
from pathlib import Path
import json
import os
import subprocess
import sys

REPOSITORY_URL = "https://github.com/adhishmodi/customer-churn-prediction.git"
GIT_BRANCH = "main"
EXPECTED_CSV = "WA_Fn-UseC_-Telco-Customer-Churn.csv"
PHASE3_SCORES = "phase_3_calibration_threshold_fold_scores.csv"

current_dir = Path.cwd()
PROJECT_ROOT = next(
    (p for p in (current_dir, *current_dir.parents)
     if (p / "src" / "final_evaluation.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    PROJECT_ROOT = Path("/content/customer-churn-prediction")
    if not (PROJECT_ROOT / "src" / "final_evaluation.py").is_file():
        if PROJECT_ROOT.exists() and (PROJECT_ROOT / ".git").is_dir():
            subprocess.run(["git", "-C", str(PROJECT_ROOT), "fetch", "origin", GIT_BRANCH], check=True)
            subprocess.run(["git", "-C", str(PROJECT_ROOT), "checkout", GIT_BRANCH], check=True)
            subprocess.run(["git", "-C", str(PROJECT_ROOT), "pull", "--ff-only", "origin", GIT_BRANCH], check=True)
        else:
            if PROJECT_ROOT.exists() and any(PROJECT_ROOT.iterdir()):
                raise RuntimeError(f"Existing project directory is not a Git checkout: {PROJECT_ROOT}")
            subprocess.run(
                ["git", "clone", "--depth", "1", "--branch", GIT_BRANCH,
                 REPOSITORY_URL, str(PROJECT_ROOT)],
                check=True,
            )

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import RAW_DATA_PATH, REPORTS_DIR, MODELS_DIR, RANDOM_SEED
from src.preprocess import clean_data, prepare_features_target, split_data
from src.train import get_imbalance_experiment_models
from src.final_evaluation import (
    evaluate_final_model, fit_final_calibrated_model, select_phase3_finalist,
)

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
DATA_PATH = RAW_DATA_PATH
PHASE3_PATH = REPORTS_DIR / PHASE3_SCORES
print(f"Project root: {PROJECT_ROOT}")
print(f"Dataset path: {DATA_PATH}")
print(f"Phase 3 fold scores: {PHASE3_PATH}")

Project root: /content/customer-churn-prediction
Dataset path: /content/customer-churn-prediction/data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv
Phase 3 fold scores: /content/customer-churn-prediction/reports/phase_3_calibration_threshold_fold_scores.csv


In [2]:
missing = []
if not DATA_PATH.is_file():
    missing.append(EXPECTED_CSV)
if not PHASE3_PATH.is_file():
    missing.append(PHASE3_SCORES)

if missing:
    try:
        from google.colab import files
    except ImportError as exc:
        raise FileNotFoundError(
            "Place these files at the paths printed above: " + ", ".join(missing)
        ) from exc
    uploaded = files.upload()
    for filename, content in uploaded.items():
        if filename == EXPECTED_CSV:
            DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
            DATA_PATH.write_bytes(content)
        elif filename == PHASE3_SCORES:
            PHASE3_PATH.parent.mkdir(parents=True, exist_ok=True)
            PHASE3_PATH.write_bytes(content)

assert DATA_PATH.is_file(), f"Missing original dataset: {DATA_PATH}"
assert PHASE3_PATH.is_file(), f"Missing Phase 3 fold scores: {PHASE3_PATH}"
print("Required files are available.")

Saving phase_3_calibration_threshold_fold_scores.csv to phase_3_calibration_threshold_fold_scores.csv
Saving WA_Fn-UseC_-Telco-Customer-Churn.csv to WA_Fn-UseC_-Telco-Customer-Churn.csv
Required files are available.


## 2. Reconstruct the original split

Recreate the same stratified 80/20 split with seed 42. Keep the test partition reserved; do not inspect its labels or calculate test metrics before the model and threshold are frozen.

In [3]:
import pandas as pd

raw_df = pd.read_csv(DATA_PATH)
cleaned_df = clean_data(raw_df)
X_all, y_all = prepare_features_target(cleaned_df)
X_train, X_test, y_train, y_test = split_data(X_all, y_all)

assert len(X_train) == 5634
assert len(X_test) == 1409
assert X_train.index.intersection(X_test.index).empty
assert y_train.index.equals(X_train.index)
assert y_test.index.equals(X_test.index)
assert set(y_train.unique()) == {0, 1}
print(f"Training rows: {len(X_train):,}")
print(f"Reserved holdout rows: {len(X_test):,}")
print("The holdout remains reserved until after the final configuration is frozen.")

Training rows: 5,634
Reserved holdout rows: 1,409
The holdout remains reserved until after the final configuration is frozen.


## 3. Select and freeze the finalist from Phase 3

Only the two XGBoost sigmoid-calibration candidates are considered. The selector validates that each candidate has exactly five outer-fold records, then ranks by mean F1, mean precision, and mean recall—in that order. The test set is not an input.

In [4]:
phase3_fold_scores = pd.read_csv(PHASE3_PATH)
selected_model_name, selection_summary = select_phase3_finalist(phase3_fold_scores)
display(selection_summary.round(6))

candidate_models = get_imbalance_experiment_models()
assert selected_model_name in ("XGBoost (balanced)", "XGBoost (unweighted)")
final_estimator = candidate_models[selected_model_name]
FINAL_CALIBRATION_METHOD = "sigmoid"

print(f"Frozen finalist: {selected_model_name}")
print(f"Frozen calibration: {FINAL_CALIBRATION_METHOD}")
print("Selection evidence: Phase 3 outer-fold training-only results.")

,model,calibration,f1_mean,f1_std,precision_mean,recall_mean
0,XGBoost (balanced),sigmoid,0.635881,0.024157,0.555974,0.747157
1,XGBoost (unweighted),sigmoid,0.634574,0.019293,0.545255,0.761204


Frozen finalist: XGBoost (balanced)
Frozen calibration: sigmoid
Selection evidence: Phase 3 outer-fold training-only results.


## 4. Fit the final model, calibrator, and threshold using training rows only

The final calibrator and F1 threshold are learned from out-of-fold probabilities generated only within X_train. Then the base pipeline is refitted on all training rows. No test features or labels are passed to this fitting function.

In [5]:
import joblib

finalized = fit_final_calibrated_model(
    X_train=X_train,
    y_train=y_train,
    estimator=final_estimator,
    calibration_method=FINAL_CALIBRATION_METHOD,
    cv_splits=5,
    random_state=RANDOM_SEED,
)

final_threshold = finalized["threshold"]
print(f"Training-only OOF-selected threshold: {final_threshold:.6f}")
print(f"Training-only OOF F1 at selected threshold: {finalized['training_oof_f1']:.6f}")
assert finalized["test_set_used_for_fitting"] is False
assert finalized["training_rows"] == len(X_train)
assert 0.0 <= final_threshold <= 1.0

artifact_path = MODELS_DIR / "final_churn_model.joblib"
joblib.dump({
    "model": finalized["model"],
    "calibrator": finalized["calibrator"],
    "calibration_method": finalized["calibration_method"],
    "threshold": finalized["threshold"],
    "training_oof_f1": finalized["training_oof_f1"],
    "selected_model_name": selected_model_name,
    "random_seed": RANDOM_SEED,
    "test_set_used_for_fitting": False,
}, artifact_path)

selection_record = {
    "selected_model": selected_model_name,
    "calibration_method": FINAL_CALIBRATION_METHOD,
    "selection_metric": "phase_3_outer_fold_f1_mean",
    "selection_summary": selection_summary.to_dict(orient="records"),
    "final_threshold": final_threshold,
    "threshold_selection_metric": "training_only_oof_f1",
    "training_oof_f1": finalized["training_oof_f1"],
    "training_rows": len(X_train),
    "test_rows": len(X_test),
    "random_seed": RANDOM_SEED,
    "test_set_used_for_fitting_or_selection": False,
    "test_set_evaluated": False,
}
selection_path = REPORTS_DIR / "phase_4_selection_record.json"
selection_path.write_text(json.dumps(selection_record, indent=2) + "\n", encoding="utf-8")
print(f"Frozen model artifact saved: {artifact_path}")
print(f"Selection record saved: {selection_path}")
print("FINAL CONFIGURATION FROZEN. Do not change the model or threshold after viewing test results.")

Training-only OOF-selected threshold: 0.344207
Training-only OOF F1 at selected threshold: 0.637266
Frozen model artifact saved: /content/customer-churn-prediction/models/final_churn_model.joblib
Selection record saved: /content/customer-churn-prediction/reports/phase_4_selection_record.json
FINAL CONFIGURATION FROZEN. Do not change the model or threshold after viewing test results.


## 5. Final held-out evaluation

This is the first Phase 4 access to the holdout labels. Evaluate the frozen model at:
- The conventional 0.50 threshold.
- The threshold selected earlier from training-only OOF predictions.

No test-set threshold search or model selection is performed.

In [6]:
test_metrics, test_predictions = evaluate_final_model(
    finalized,
    X_test,
    y_test,
)
display(test_metrics.round(4))

assert len(test_predictions) == 2 * len(y_test)
assert set(test_metrics["threshold_strategy"]) == {"default_0.50", "training_oof_f1"}
assert test_metrics["threshold"].between(0.0, 1.0).all()
assert test_metrics[["accuracy", "precision", "recall", "f1", "roc_auc",
                     "average_precision", "brier_score", "log_loss"]].notna().all().all()

,threshold_strategy,threshold,accuracy,precision,recall,f1,roc_auc,average_precision,brier_score,log_loss,true_negative,false_positive,false_negative,true_positive
0,default_0.50,0.5000,0.8084,0.6656,0.5588,0.6076,0.8455,0.663,0.1356,0.4156,930,105,165,209
1,training_oof_f1,0.3442,0.7658,0.5438,0.7299,0.6233,0.8455,0.663,0.1356,0.4156,806,229,101,273


## 6. Compare with the frozen Milestone 4 baseline

These historical baseline values were recorded before Phase 3. They are included as a fixed reference and must not be recomputed or tuned against the holdout.

In [7]:
historical_baseline = pd.DataFrame([
    {
        "model": "Milestone 4 XGBoost baseline",
        "threshold_strategy": "historical_default_0.50",
        "threshold": 0.50,
        "accuracy": 0.7402, "precision": 0.5067, "recall": 0.8102,
        "f1": 0.6235, "roc_auc": 0.8463, "average_precision": 0.6660,
        "true_negative": 740, "false_positive": 295,
        "false_negative": 71, "true_positive": 303,
    },
    {
        "model": "Milestone 4 XGBoost baseline",
        "threshold_strategy": "historical_training_oof_threshold_0.53",
        "threshold": 0.53,
        "accuracy": 0.7509, "precision": 0.5202, "recall": 0.7914,
        "f1": 0.6278, "roc_auc": 0.8463, "average_precision": 0.6660,
        "true_negative": None, "false_positive": None,
        "false_negative": None, "true_positive": None,
    },
])
final_comparison = test_metrics.copy()
final_comparison.insert(0, "model", selected_model_name + " + sigmoid")
comparison = pd.concat([historical_baseline, final_comparison], ignore_index=True, sort=False)
display(comparison.round(4))

,model,threshold_strategy,threshold,accuracy,precision,recall,f1,roc_auc,average_precision,true_negative,false_positive,false_negative,true_positive,brier_score,log_loss
0,Milestone 4 XGBoost baseline,historical_default_0.50,0.5000,0.7402,0.5067,0.8102,0.6235,0.8463,0.666,740.0,295.0,71.0,303.0,NaN,NaN
1,Milestone 4 XGBoost baseline,historical_training_oof_threshold_0.53,0.5300,0.7509,0.5202,0.7914,0.6278,0.8463,0.666,NaN,NaN,NaN,NaN,NaN,NaN
2,XGBoost (balanced) + sigmoid,default_0.50,0.5000,0.8084,0.6656,0.5588,0.6076,0.8455,0.663,930.0,105.0,165.0,209.0,0.1356,0.4156
3,XGBoost (balanced) + sigmoid,training_oof_f1,0.3442,0.7658,0.5438,0.7299,0.6233,0.8455,0.663,806.0,229.0,101.0,273.0,0.1356,0.4156


## 7. Save final evaluation outputs and close the holdout

Save the metrics, predictions, and completed selection record. From this point onward, do not change the model, calibrator, or threshold based on these test results. Any further experiments must use training-only validation or a newly collected external test set.

In [8]:
metrics_path = REPORTS_DIR / "phase_4_final_test_metrics.csv"
predictions_path = REPORTS_DIR / "phase_4_final_test_predictions.csv"
comparison_path = REPORTS_DIR / "phase_4_baseline_comparison.csv"

test_metrics.insert(0, "model", selected_model_name + " + sigmoid")
test_metrics.to_csv(metrics_path, index=False)
test_predictions.to_csv(predictions_path, index=False)
comparison.to_csv(comparison_path, index=False)

selection_record["test_set_evaluated"] = True
selection_record["test_rows_evaluated"] = len(y_test)
selection_record["test_evaluation_files"] = [
    str(metrics_path), str(predictions_path), str(comparison_path)
]
selection_path.write_text(json.dumps(selection_record, indent=2) + "\n", encoding="utf-8")

print(f"Test rows evaluated: {len(y_test):,}")
print(f"Test churn prevalence: {y_test.mean():.2%}")
print(f"Final test metrics: {metrics_path}")
print(f"Prediction records: {predictions_path}")
print(f"Baseline comparison: {comparison_path}")
print("Phase 4 evaluation completed. The holdout is now considered spent.")

Test rows evaluated: 1,409
Test churn prevalence: 26.54%
Final test metrics: /content/customer-churn-prediction/reports/phase_4_final_test_metrics.csv
Prediction records: /content/customer-churn-prediction/reports/phase_4_final_test_predictions.csv
Baseline comparison: /content/customer-churn-prediction/reports/phase_4_baseline_comparison.csv
Phase 4 evaluation completed. The holdout is now considered spent.


## Final interpretation and completion checklist

- [x] Finalist selected from Phase 3 outer-fold training-only results.
- [x] Sigmoid calibrator and threshold fitted from training-only OOF predictions.
- [x] Final base pipeline fitted on all 5,634 training rows.
- [x] Holdout evaluated only after the configuration was frozen.
- [x] No test-set threshold search, tuning, or model selection performed.
- [x] Historical Milestone 4 baseline values preserved for comparison.
- [x] Executed notebook and aggregate metric/comparison outputs committed to GitHub.

The three CSV exports were generated in the Colab `reports/` directory, which is ignored by Git; they are not committed to the repository. The row-level prediction CSV was not uploaded because it is not needed for the aggregate final report. The notebook's saved outputs contain the metrics and baseline comparison used in the report.

**Final interpretation:** at threshold 0.344207, the final model achieved 76.58% accuracy, 54.38% precision, 72.99% recall, 62.33% F1, 84.55% ROC-AUC, and 66.30% Average Precision. Compared with the historical threshold-0.53 baseline, accuracy and precision are higher, while recall, F1, ROC-AUC, and Average Precision are lower. Report this as a precision-recall trade-off, not an overall improvement.

**Holdout closure:** the same holdout was used for the Milestone 4 baseline and this Phase 4 comparison. Do not use it for any further tuning, feature selection, calibration, threshold selection, or model selection. Future experiments must use training-only validation or a genuinely new external dataset.